# Phase 4 — Data visualization and analysis (EDA)

Eyeball the data before starting model development.
Reads `CURATED.SENSORS` / `IMAGES_STAGE` from a Snowflake Notebook (Container Runtime).

**What this notebook does, cell by cell:**

1. Load every labeled frame's sensor row into a local pandas DataFrame (cell 2).
2. Plot a histogram of the distance label, to check near-range coverage (cell 4).
3. Preview a handful of dashcam thumbnails alongside their radar distance (cell 6).
4. Compute each sensor channel's correlation with distance, and confirm the known
   `speed` +0.63 spurious correlation reproduces (cells 8-9) — this is the concrete
   evidence used later to justify *excluding* `speed` from the sensor-only model's
   inputs (Phase 6/7).
5. Plot the distribution of per-segment valid-label rate, to spot any unusually
   low-quality segments (cell 11).

This notebook is read-only and purely exploratory — nothing here blocks or gates
later phases; it exists so a human can sanity-check the data visually before
trusting it for modeling.


### Cell: load all labeled sensor rows into pandas

Sets the session's database/schema to `CAR_MULTIMODAL_AI_DB.CURATED` (same reason
as Phase 3 — a fresh Notebook session doesn't inherit it automatically), then runs
`SELECT * FROM SENSORS WHERE DISTANCE IS NOT NULL` and materializes the result as a
pandas DataFrame `df` (every later cell in this notebook reuses `df`). Frames
without a valid label are filtered out here because they have no `DISTANCE` to
visualize or correlate against. The cell's output, `df.shape`, is a
`(row_count, column_count)` tuple — `row_count` should match the "valid" count from
Phase 3's criterion 3 (17,516 for the default Chunk_1 run).


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import plotly.express as px

session = get_active_session()
session.sql("USE DATABASE CAR_MULTIMODAL_AI_DB").collect()
session.sql("USE SCHEMA CURATED").collect()
df = session.sql("SELECT * FROM SENSORS WHERE DISTANCE IS NOT NULL").to_pandas()
df.shape

## 1. Histogram of the distance label (is there enough near-range data?)

### Cell: plot the DISTANCE histogram

Draws a Plotly histogram of `DISTANCE` (60 bins) and calls `fig.show()` to render it
inline in the Notebook. Look at the shape: real driving data should show most
frames clustered in the 20-80m range (typical highway following distance), with a
longer tail out toward 150m and a smaller but non-empty bucket below 20m (close
following / stop-and-go traffic). **A distribution with almost no mass below 30m**
would be a red flag for the ablation study later — Phase 7's "0-30m banded MAE"
metric specifically needs enough near-range examples to be meaningful.


In [ ]:
fig = px.histogram(df, x="DISTANCE", nbins=60, title="Distribution of lead-vehicle distance [m]")
fig.show()

## 2. Thumbnail gallery + corresponding radar distance

Fetch a handful of images from `IMAGES_STAGE/full/` via presigned URLs and display them
alongside their distance. ⚠ This is public-road dashcam footage. Always blur it before
zooming in or including it in slides.

### Cell: sample 8 frames and generate viewable image URLs

Two separate things happen here:

1. `df.sample(8, random_state=42)` picks 8 random rows from the DataFrame loaded in
   cell 2 (the fixed `random_state` makes the sample reproducible run-to-run) and
   keeps just `SAMPLE_ID`, `SEGMENT`, `DISTANCE` — this is *not yet* the images, just
   which 8 samples to look at.
2. A separate SQL query lists **every** file under `IMAGES_STAGE/full/` via
   `DIRECTORY(@IMAGES_STAGE)` and calls `GET_PRESIGNED_URL()` on each one, producing
   a temporary, time-limited HTTPS URL that can render the image directly (e.g. by
   pasting it into a browser or an `<img>` tag) without needing a Snowflake session.

The cell's displayed output is just the `sample` table (8 rows of IDs + distances);
matching a `SAMPLE_ID` to a `RELATIVE_PATH` in the `urls` table and opening that URL
is a manual follow-up step, not something the cell does automatically. **Before
sharing any rendered image outside this notebook, blur it** — comma2k19 is
unmodified public-road footage and may contain identifiable plates/faces/people.


In [ ]:
sample = df.sample(8, random_state=42)[["SAMPLE_ID", "SEGMENT", "DISTANCE"]]
urls = session.sql(
    "SELECT RELATIVE_PATH, GET_PRESIGNED_URL(@IMAGES_STAGE, 'full/' || RELATIVE_PATH) AS URL "
    "FROM DIRECTORY(@IMAGES_STAGE) WHERE RELATIVE_PATH LIKE 'full/%'"
).to_pandas()
sample

## 3. Sensor x distance correlation check

**Reproduce the `speed` +0.63 correlation**. This is the
evidence for "why `speed` is excluded from the sensor branch's training inputs" (used in
Phase 6/7).

### Cell: correlation of 9 sensor channels against DISTANCE

Computes the Pearson correlation coefficient between `DISTANCE` and each of 9 raw
sensor channels (`SPEED`, `STEERING_ANGLE`, `WHEEL_SPEED_MEAN`, 3-axis `ACCEL_*`,
3-axis `GYRO_*`), using pandas' `.corr()` on the joined DataFrame, then sorts the
result ascending. The printed output is a small Series: channel name → correlation
(a number between -1 and +1).

**Why `SPEED` stands out:** ego vehicle speed correlates with lead-vehicle distance
not because the car can sense distance from its own speed, but because of traffic
flow — in heavy/slow traffic the car ahead is typically close, and at highway speed
it's typically far. That's a real statistical pattern in the data, but it's a
**confound, not a causal signal** the model should be allowed to exploit (it would
make the "sensor-only can't solve this" ablation finding look artificially good).
`WHEEL_SPEED_MEAN` shows a nearly identical correlation to `SPEED` for the same
reason (they measure almost the same physical quantity).


In [ ]:
channels = ["SPEED", "STEERING_ANGLE", "WHEEL_SPEED_MEAN",
            "ACCEL_X", "ACCEL_Y", "ACCEL_Z", "GYRO_X", "GYRO_Y", "GYRO_Z"]
corr = df[channels + ["DISTANCE"]].corr()["DISTANCE"].drop("DISTANCE").sort_values()
corr

### Cell: visualize the correlations and assert the SPEED finding holds

Plots the same correlation Series as a horizontal bar chart, with a red dashed
reference line at the documented `+0.63` value, then
`assert corr["SPEED"] > 0.5` — a looser check than "exactly 0.63" on purpose, since
this is real data and the exact number will drift slightly between chunks/reruns.
**If this assertion fails** (i.e. `SPEED`'s correlation with distance drops well
below 0.5), it's a signal that Phase 2's label generation is producing something
different from the 45-minute version's original data — worth investigating before
trusting Phase 6/7's decision to exclude `speed` from the sensor model's inputs.


In [ ]:
fig = px.bar(corr, orientation="h", title="Correlation: sensor channel x lead-vehicle distance")
fig.add_vline(x=0.63, line_dash="dash", line_color="red",
              annotation_text="known speed correlation +0.63")
fig.show()
assert corr["SPEED"] > 0.5, "speed-distance correlation is weaker than expected (+0.63). Check label generation."

## 4. Valid-label rate per segment (outlier/quality detection)

### Cell: per-segment label coverage, plus the worst 10 segments

Groups by `SEGMENT` and computes, for each of the (up to 188) driving segments,
`N_FRAMES` (total labeled-attempt frames) and `N_VALID` (frames where a lead
vehicle was actually found), then `VALID_RATE = 100 * N_VALID / N_FRAMES`. It plots
a histogram of `VALID_RATE` across all segments, then prints a table of the 10
segments with the *lowest* valid-label rate (sorted ascending, `.head(10)`).

**How to read this:** most segments should cluster somewhere in the 60-100% range;
a segment near 0% valid rate typically means "no lead vehicle observed for most of
that minute" (e.g. an open road with no traffic ahead) rather than a data bug —
but if *many* segments show 0%, that's worth investigating as a possible label
matching problem rather than assuming it's all just empty roads.


In [ ]:
seg_rate = (
    session.sql(
        "SELECT SEGMENT, "
        "       COUNT(*) AS N_FRAMES, "
        "       SUM(IFF(DISTANCE IS NOT NULL, 1, 0)) AS N_VALID "
        "FROM SENSORS GROUP BY SEGMENT"
    )
    .to_pandas()
)
seg_rate["VALID_RATE"] = 100 * seg_rate["N_VALID"] / seg_rate["N_FRAMES"]
fig = px.histogram(seg_rate, x="VALID_RATE", nbins=30,
                   title="Distribution of valid-label rate per segment")
fig.show()
seg_rate.sort_values("VALID_RATE").head(10)  # inspect the lowest-quality segments

Once EDA is complete, move on to Phase 5 (embedding extraction) / Phase 6
(small-scale model validation).
